In [117]:
import pandas as pd
import numpy as np
import pickle

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [118]:
df = pd.read_csv("dice_com-job_us_sample.csv")

print(df.head())
print(df.shape)

                                       advertiserurl  \
0  https://www.dice.com/jobs/detail/AUTOMATION-TE...   
1  https://www.dice.com/jobs/detail/Information-S...   
2  https://www.dice.com/jobs/detail/Business-Solu...   
3  https://www.dice.com/jobs/detail/Java-Develope...   
4  https://www.dice.com/jobs/detail/DevOps-Engine...   

                             company  \
0  Digital Intelligence Systems, LLC   
1  University of Chicago/IT Services   
2               Galaxy Systems, Inc.   
3                      TransTech LLC   
4                   Matrix Resources   

                            employmenttype_jobstatus  \
0  C2H Corp-To-Corp, C2H Independent, C2H W2, 3 M...   
1                                          Full Time   
2                                          Full Time   
3                                          Full Time   
4                                          Full Time   

                                      jobdescription               jobid  \
0  Lookin

In [86]:
print(df.columns)

Index(['advertiserurl', 'company', 'employmenttype_jobstatus',
       'jobdescription', 'jobid', 'joblocation_address', 'jobtitle',
       'postdate'],
      dtype='str')


In [119]:
df = df[['jobtitle', 'jobdescription']].dropna()

df.dropna(inplace=True)

print(df.head())

                                            jobtitle  \
0                           AUTOMATION TEST ENGINEER   
1                      Information Security Engineer   
2                       Business Solutions Architect   
3  Java Developer (mid level)- FT- GREAT culture,...   
4                                    DevOps Engineer   

                                      jobdescription  
0  Looking for Selenium engineers...must have sol...  
1  The University of Chicago has a rapidly growin...  
2  GalaxE.SolutionsEvery day, our solutions affec...  
3  Java DeveloperFull-time/direct-hireBolingbrook...  
4  Midtown based high tech firm has an immediate ...  


In [120]:
encoder = LabelEncoder()

df['label'] = encoder.fit_transform(df['jobdescription'])

print(df.head())

                                            jobtitle  \
0                           AUTOMATION TEST ENGINEER   
1                      Information Security Engineer   
2                       Business Solutions Architect   
3  Java Developer (mid level)- FT- GREAT culture,...   
4                                    DevOps Engineer   

                                      jobdescription  label  
0  Looking for Selenium engineers...must have sol...   3596  
1  The University of Chicago has a rapidly growin...   7214  
2  GalaxE.SolutionsEvery day, our solutions affec...   2132  
3  Java DeveloperFull-time/direct-hireBolingbrook...   2830  
4  Midtown based high tech firm has an immediate ...   3747  


In [122]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(df['jobtitle'])

X = tokenizer.texts_to_sequences(df['jobtitle'])

X = pad_sequences(X, maxlen=100)

y = df['label']

In [123]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(y_train.shape)

(7200, 100)
(7200,)


In [124]:
model = Sequential([
    Embedding(
        input_dim=5000,
        output_dim=64
    ),
    LSTM(64),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(len(df['label'].unique()), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_6 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [125]:
history = model.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 14s 51ms/step - accuracy: 0.0000e+00 - loss: 9.1123 - val_accuracy: 0.0000e+00 - val_loss: 9.1718
Epoch 2/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 9s 42ms/step - accuracy: 2.7778e-04 - loss: 9.0489 - val_accuracy: 0.0000e+00 - val_loss: 9.3338
Epoch 3/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 10s 42ms/step - accuracy: 6.9444e-04 - loss: 8.9746 - val_accuracy: 5.5556e-04 - val_loss: 9.6346
Epoch 4/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 11s 49ms/step - accuracy: 0.0021 - loss: 8.4794 - val_accuracy: 5.5556e-04 - val_loss: 10.7757
Epoch 5/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 9s 38ms/step - accuracy: 0.0033 - loss: 7.8135 - val_accuracy: 5.5556e-04 - val_loss: 12.3186
Epoch 6/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 9s 40ms/step - accuracy: 0.0071 - loss: 7.2291 - val_accuracy: 0.0011 - val_loss: 13.8309
Epoch 7/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - accuracy: 0.0128 - loss: 6.6489 - val_accuracy: 0.0022 - val_loss: 15.3851
Epoch 8/10
225/225 ━━━━━━━━━━━━━━━━━━━━ 8s 35ms/step - acc

In [126]:
model.save("job_recommendation_model.h5")

In [127]:
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [128]:
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)

In [129]:
text = ["python machine learning sql pandas"]

seq = tokenizer.texts_to_sequences(text)

pad = pad_sequences(seq, maxlen=100)

pred = model.predict(pad)

job = encoder.inverse_transform([np.argmax(pred)])

print("Recommended Job:", job[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 221ms/step
Recommended Job: Consultant Needed for a Global Firm for a 6 month contract. Position can be worked out of Charlotte, North Carolina or alternatively, Syracuse, NY. This role is to ensure the stability, integrity, and efficient operation of the IPT environment and Call Center applications.  This is achieved by optimizing network software and associated operating systems as well as developing, coding, testing and debugging new software or enhancements to existing software.  Candidates must have good understanding of contact center applications and work with technical staff to provide Tier III support to resolve problems and develop prevention plans as needed. Strong working knowledge and hands-on experience in a Cisco IPT environment.Must have experience with Cisco Communication Manager, voice gateways, QOS, SIP, and Cisco Unity Connection.  Some working knowledge of the other traditional telephony/telecom system is also a must.5+ years of applicab